In [0]:
start_time = '2026-05-29 14:00:00'
end_time = '2026-05-29 15:00:00'

In [0]:
spark.sql(
    f"""
    WITH viewing_content AS (
        SELECT c.fk_tvid, c.session_start, c.session_end
        , NULLIF(REGEXP_REPLACE(tuner_tms_show.epi_title, '[\“\”\"\^\@\,]', ''), '') AS tuner_tms_epi_title
        , NULLIF(REGEXP_REPLACE(tuner_tivo_show.epi_title, '[\“\”\"\^\@\,]', ''), '') AS tuner_tivo_epi_title
        , tuner_tivo_show.series_id AS tuner_series_id
        FROM prod.detection.viewing_content_firehose AS c
        JOIN detection.epg_show AS tuner_tivo_show
        ON tuner_tivo_show.show_id = c.tuner_program_id
        AND tuner_tivo_show.vendor_name = 'TIVO'
        JOIN detection.epg_show AS tuner_tms_show
        ON tuner_tms_show.show_id = c.tms_tuner_program_id
        AND tuner_tms_show.vendor_name = 'TMS'
        WHERE c.session_start >= '{start_time}'
        AND c.session_start <= '{end_time}'
        AND c.partition_key >= DATE('{start_time}')
        AND c.partition_key <= DATE('{start_time}')
    )
    , content_golden AS (
        SELECT c.fk_tvid, c.session_start, c.session_end, tuner_tms_epi_title, tuner_tivo_epi_title, tuner_series_id
        FROM dev.detection.viewing_content_golden AS c
        WHERE c.session_start >= '{start_time}'
        AND c.session_start <= '{end_time}'
        AND c.session_start_hour >= '{start_time}'
        AND c.session_start_hour <= '{end_time}'
    )
    , validity_check AS (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        , vc.tuner_tms_epi_title <=> cg.tuner_tms_epi_title AS tms_title_match
        , vc.tuner_tivo_epi_title <=> cg.tuner_tivo_epi_title AS tivo_title_match
        , vc.tuner_series_id <=> cg.tuner_series_id AS series_id_match
        FROM viewing_content AS vc
        JOIN content_golden AS cg
          ON vc.fk_tvid = cg.fk_tvid
         AND vc.session_start = cg.session_start
         AND vc.session_end = cg.session_end
    )
    SELECT tms_title_match, tivo_title_match, series_id_match
    , COUNT(*) AS session_count
    FROM validity_check
    GROUP BY 1, 2, 3
;
""").display()

In [0]:
spark.sql(
    f"""
    WITH viewing_content AS (
        SELECT c.fk_tvid, c.session_start, c.session_end
        , NULLIF(REGEXP_REPLACE(tuner_tms_show.epi_title, '[\“\”\"\^\@\,]', ''), '') AS tuner_tms_epi_title
        , NULLIF(REGEXP_REPLACE(tuner_tivo_show.epi_title, '[\“\”\"\^\@\,]', ''), '') AS tuner_tivo_epi_title
        , tuner_tivo_show.series_id AS tuner_series_id
        FROM prod.detection.viewing_content_firehose AS c
        JOIN detection.epg_show AS tuner_tivo_show
        ON tuner_tivo_show.show_id = c.tuner_program_id
        AND tuner_tivo_show.vendor_name = 'TIVO'
        JOIN detection.epg_show AS tuner_tms_show
        ON tuner_tms_show.show_id = c.tms_tuner_program_id
        AND tuner_tms_show.vendor_name = 'TMS'
        WHERE c.session_start >= '{start_time}'
        AND c.session_start <= '{end_time}'
        AND c.partition_key >= DATE('{start_time}')
        AND c.partition_key <= DATE('{start_time}')
    )
    , content_golden AS (
        SELECT c.fk_tvid, c.session_start, c.session_end, tuner_tms_epi_title, tuner_tivo_epi_title, tuner_series_id
        FROM dev.detection.viewing_content_golden AS c
        WHERE c.session_start >= '{start_time}'
        AND c.session_start <= '{end_time}'
        AND c.session_start_hour >= '{start_time}'
        AND c.session_start_hour <= '{end_time}'
    )
    , validity_check AS (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        , vc.tuner_tms_epi_title AS viewing_content_tms_epi_title
        , cg.tuner_tms_epi_title AS dev_golden_epi_title
        , vc.tuner_tivo_epi_title AS viewing_content_tivo_epi_title
        , cg.tuner_tivo_epi_title AS dev_golden_tivo_epi_title
        , vc.tuner_series_id AS viewing_content_series_id
        , cg.tuner_series_id AS dev_golden_series_id
        , vc.tuner_tms_epi_title <=> cg.tuner_tms_epi_title AS tms_title_match
        , vc.tuner_tivo_epi_title <=> cg.tuner_tivo_epi_title AS tivo_title_match
        , vc.tuner_series_id <=> cg.tuner_series_id AS series_id_match
        FROM viewing_content AS vc
        JOIN content_golden AS cg
          ON vc.fk_tvid = cg.fk_tvid
         AND vc.session_start = cg.session_start
         AND vc.session_end = cg.session_end
    )
    SELECT *
    FROM validity_check
    WHERE NOT tms_title_match OR NOT tivo_title_match OR NOT series_id_match
    ORDER BY 1, 2, 3
;
""").display()